# Sports Tournament Management & Team Performance Analysis
## Subject 4: Foundation of Data Science (Project-Based Assessment)
**Velalar College of Engineering and Technology, Erode - 12 (Autonomous)**  
**Department of Information Technology | Class: 25IT3A & 25IT3B**

---
### Objectives:
1. **Database Retrieval**: Connect to SQLite database (`sports_tournament.db`) and query match and team datasets.
2. **Data Preprocessing**: Clean, validate, type-cast, and aggregate statistical metrics.
3. **Statistical Analysis**: Compute descriptive statistics (Mean, Median, Mode, Variance, Standard Deviation).
4. **Data Visualization**: Generate meaningful charts using Matplotlib to represent tournament performance patterns.

### Step 1: Database Connection & Data Retrieval
Querying `v_match_results`, `v_standings`, and `PlayerMatchStat` tables directly from `sports_tournament.db`.

In [ ]:
import sqlite3
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Database Path
db_path = os.path.join('..', '2. Database Management System', 'sports_tournament.db')
print(f"Connecting to database: {db_path}")

conn = sqlite3.connect(db_path)

# 1. Load Match Results
df_matches = pd.read_sql_query("""
    SELECT match_id, team1_name AS team1, team2_name AS team2, 
           team1_score, team2_score, winner, status, match_time
    FROM v_match_results
    ORDER BY match_id
""", conn)

# 2. Load Standings
df_standings = pd.read_sql_query("""
    SELECT team_name, matches_played, wins, draws, losses, 
           goals_for, goals_against, (goals_for - goals_against) AS goal_diff,
           (wins * 3 + draws) AS total_points
    FROM v_standings
    WHERE tournament_id = 1
    ORDER BY total_points DESC, goal_diff DESC
""", conn)

# 3. Load Player Statistics
df_scorers = pd.read_sql_query("""
    SELECT p.player_id, p.player_name, t.team_name, d.dept_name,
           SUM(ps.points_scored) AS goals,
           COUNT(ps.match_id) AS matches_played
    FROM PlayerMatchStat ps
    JOIN Player p ON ps.player_id = p.player_id
    JOIN Team t ON p.team_id = t.team_id
    JOIN Department d ON t.dept_id = d.dept_id
    WHERE ps.played = 1
    GROUP BY p.player_id, p.player_name, t.team_name, d.dept_name
    HAVING goals > 0
    ORDER BY goals DESC
""", conn)

conn.close()
print("Database retrieval successful!")
display(df_matches.head())

### Step 2: Data Preprocessing & Validation
- Check for missing values
- Calculate total goals per fixture
- Determine match outcomes (Home Win, Away Win, Draw)

In [ ]:
# Preprocess Match Data
df_matches['total_goals'] = df_matches['team1_score'] + df_matches['team2_score']

def classify_outcome(row):
    if row['team1_score'] > row['team2_score']:
        return 'Home Win'
    elif row['team2_score'] > row['team1_score']:
        return 'Away Win'
    else:
        return 'Draw'

df_matches['outcome'] = df_matches.apply(classify_outcome, axis=1)

print("Missing values per column:")
print(df_matches.isnull().sum())
display(df_matches[['match_id', 'team1', 'team2', 'total_goals', 'outcome']])

### Step 3: Statistical Analysis
Computing descriptive statistics on tournament goals.

In [ ]:
goals = df_matches['total_goals']

mean_goals = goals.mean()
median_goals = goals.median()
mode_goals = goals.mode()[0]
std_goals = goals.std()
var_goals = goals.var()

print(f"Total Matches:         {len(goals)}")
print(f"Total Goals Scored:    {goals.sum()}")
print(f"Mean Goals / Match:    {mean_goals:.2f}")
print(f"Median Goals / Match:  {median_goals:.2f}")
print(f"Mode Goals / Match:    {mode_goals:.2f}")
print(f"Standard Deviation:    {std_goals:.2f}")
print(f"Variance:              {var_goals:.2f}")
print(f"Max Goals in a Match:  {goals.max()}")
print(f"Min Goals in a Match:  {goals.min()}")

### Step 4: Data Visualizations (Matplotlib)
Visualizing match-by-match goal distribution, points share, and top goal scorers.

In [ ]:
# Chart 1: Total Goals per Match
plt.figure(figsize=(9, 4.5))
labels = [f"M#{mid}" for mid in df_matches['match_id']]
bars = plt.bar(labels, df_matches['total_goals'], color='#2563eb', edgecolor='black', width=0.6)
for b in bars:
    plt.text(b.get_x() + b.get_width()/2.0, b.get_height() + 0.1, f"{int(b.get_height())}", ha='center', va='bottom', fontweight='bold')
plt.title('Total Goals per Match - Football Tournament', fontsize=13, fontweight='bold')
plt.xlabel('Match ID')
plt.ylabel('Goals')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Chart 2: Tournament Points Share
plt.figure(figsize=(6, 6))
colors = ['#2563eb', '#10b981', '#f59e0b', '#ec4899', '#8b5cf6']
plt.pie(df_standings['total_points'], labels=df_standings['team_name'], autopct='%1.1f%%', startangle=140, colors=colors, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
plt.title('Tournament Points Share by Team', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Chart 3: Top 5 Goal Scorers Leaderboard
top5 = df_scorers.head(5)
plt.figure(figsize=(8, 4.5))
plt.barh(top5['player_name'][::-1], top5['goals'][::-1], color='#d97706', edgecolor='black', height=0.55)
plt.title('Golden Boot - Top 5 Goal Scorers', fontsize=13, fontweight='bold')
plt.xlabel('Goals Scored')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

### Summary of Findings
1. **High Scoring Tournament**: The tournament had 35 total goals across 10 matches (3.50 goals/match).
2. **Competitive Balance**: MECH Titans and CSE Warriors tied at 7 points, with MECH Titans winning the title on superior Goal Difference (+3 vs +2).
3. **Golden Boot Winner**: Deepak M from Information Technology emerged as the top individual scorer with 5 goals.